# 📡 Modulation Classification — Attention+CLDNN (All-SNR, AWGN, Ideal Conditions)

**One model. Trained on the full dataset (all SNRs, natural AWGN, no extra impairments). Evaluated at every SNR, with a clear callout at 0 dB.**

```
[Data: RML2016.10b — all SNRs, AWGN as-is]
        │
        ▼
  Train / Val / Test split (stratified)
        │
        ▼
  Attention+CLDNN training
        │
        ▼
  Accuracy vs SNR curve  +  Accuracy @ 0 dB  +  Confusion matrix @ 0 dB
```

No cross-SNR generalisation experiment (train-high/test-low), no colored/impulsive/Laplacian noise, no CFO/fading. Just: train on everything, see how it performs, especially at 0 dB.

## ⚙️ Step 1 — Install & Kaggle Setup

In [ ]:
!pip install kaggle -q

from google.colab import files
import os

if not os.path.exists("/root/.kaggle/kaggle.json"):
    uploaded = files.upload()
    for fn in uploaded.keys():
        !mkdir -p ~/.kaggle
        !mv {fn} ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle API key uploaded.")
else:
    print("✅ Kaggle API key already present.")

## 📦 Step 2 — Download Dataset (RML2016.10b)

In [ ]:
!kaggle datasets download -d marwanabudeeb/rml201610b -q
!unzip -o rml201610b.zip -d "./radioml_10b" -q
print("✅ Dataset ready at ./radioml_10b/")

## 📊 Step 3 — Load & Parse the Full Dataset (all SNRs, AWGN as-is)

In [ ]:
import pickle, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# ── Load raw dataset ──────────────────────────────────────────────────────
data_path = './radioml_10b/RML2016.10b.dat'
print(f"Loading: {data_path}")
with open(data_path, 'rb') as f:
    u = pickle._Unpickler(f); u.encoding = 'latin1'
    raw = u.load()
print("✅ Loaded.")

# ── Parse into arrays (every (mod, SNR) pair — no filtering) ──────────────
X_list, Y_list, SNR_list, mods = [], [], [], []
for (mod, snr) in raw.keys():
    samples = raw[(mod, snr)]          # (1000, 2, 128)
    X_list.append(samples)
    Y_list  += [mod] * samples.shape[0]
    SNR_list+= [snr] * samples.shape[0]
    if mod not in mods: mods.append(mod)

X_raw   = np.vstack(X_list)           # (N, 2, 128)
SNR_raw = np.array(SNR_list)
Y_str   = np.array(Y_list)

# (N,2,128) → (N,128,2)  [samples, timesteps, IQ]
X_raw = np.transpose(X_raw, (0, 2, 1))

enc       = LabelEncoder()
Y_int_raw = enc.fit_transform(Y_str)
Y_raw     = to_categorical(Y_int_raw)
n_classes = Y_raw.shape[1]

print(f"Total samples : {X_raw.shape[0]:,}")
print(f"Input shape   : {X_raw.shape[1:]}")
print(f"Classes ({n_classes})   : {mods}")
print(f"SNR range     : {sorted(np.unique(SNR_raw))} dB")
print("\nNote: RML2016.10b samples already contain AWGN at their labeled SNR.")
print("We train on ALL of them together — no filtering, no extra synthetic impairments.")

## 🏗️ Step 4 — Model Definition (Attention+CLDNN)

In [ ]:
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, BatchNormalization,
                                      LSTM, Dense, Dropout, Input)
from tensorflow.keras.models import Model, Sequential

# ══ Temporal Attention Layer ══════════════════════════════════════════════
class TemporalAttention(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(input_shape[1],  1),
                                 initializer='zeros',         trainable=True)
        super().build(input_shape)

    def call(self, x):
        score   = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        weights = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

    def get_config(self):
        return super().get_config()

def _conv_block(model, filters, dropout=True):
    model.add(Conv1D(filters, 3, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2))
    if dropout: model.add(Dropout(0.5))

def build_attention_cldnn(n_classes, timesteps=128, features=2):
    inp = Input(shape=(timesteps, features))
    x = inp
    for filt, drop in [(128, True), (128, True), (64, False)]:
        x = Conv1D(filt, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        if drop: x = Dropout(0.5)(x)
    x = LSTM(128, return_sequences=True)(x);  x = Dropout(0.5)(x)
    x = LSTM(128, return_sequences=True)(x);  x = Dropout(0.5)(x)
    x = TemporalAttention()(x)
    x = Dense(256, activation='relu')(x); x = Dropout(0.5)(x)
    out = Dense(n_classes, activation='softmax')(x)
    m = Model(inp, out, name="Attention_CLDNN")
    m.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return m

print("✅ Attention+CLDNN builder ready.")

## 🔀 Step 5 — Train / Val / Test Split (ALL SNRs together)

One stratified split across the *entire* dataset — every SNR from −20 dB to +18 dB is present
in train, val, and test. Normalisation statistics come from the training set only.

In [ ]:
RANDOM_STATE = 42

Xtr, Xtmp, Ytr, Ytmp, SNRtr, SNRtmp = train_test_split(
    X_raw, Y_raw, SNR_raw,
    test_size=0.30, random_state=RANDOM_STATE, stratify=Y_int_raw)

Xval, Xte, Yval, Yte, SNRval, SNRte = train_test_split(
    Xtmp, Ytmp, SNRtmp,
    test_size=0.50, random_state=RANDOM_STATE, stratify=np.argmax(Ytmp, axis=1))

# Normalise using TRAIN stats only
mu  = Xtr.reshape(-1, Xtr.shape[-1]).mean(0)
sig = Xtr.reshape(-1, Xtr.shape[-1]).std(0) + 1e-9

Xtr_n  = (Xtr  - mu) / sig
Xval_n = (Xval - mu) / sig
Xte_n  = (Xte  - mu) / sig

print(f"Train: {Xtr_n.shape[0]:,}  Val: {Xval_n.shape[0]:,}  Test: {Xte_n.shape[0]:,}")
print(f"SNRs present in train: {sorted(np.unique(SNRtr))}")

## 🚀 Step 6 — Train Attention+CLDNN on ALL SNRs

In [ ]:
model = build_attention_cldnn(n_classes)
model.summary()

cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1)

history = model.fit(
    Xtr_n, Ytr,
    validation_data=(Xval_n, Yval),
    epochs=30, batch_size=256,
    callbacks=[cb], verbose=1)

print("\n✅ Training complete.")

In [ ]:
# ── Save checkpoint (optional, useful if running on Colab) ──────────────
import os
CKPT_DIR = '/content/drive/MyDrive/modclass_checkpoints' if os.path.exists('/content/drive') else './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
model.save(f'{CKPT_DIR}/attn_cldnn_allsnr.keras')
print(f"✅ Model saved to {CKPT_DIR}/attn_cldnn_allsnr.keras")

## 📈 Step 7 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=.3)

plt.suptitle('Attention+CLDNN — Trained on All SNRs (AWGN, ideal)', fontweight='bold')
plt.tight_layout(); plt.show()

## 🎯 Step 8 — Overall Test Accuracy & Accuracy vs SNR

In [ ]:
# ── Overall accuracy on held-out test set (all SNRs mixed) ────────────────
test_loss, test_acc = model.evaluate(Xte_n, Yte, verbose=0)
print(f"Overall test accuracy (all SNRs): {test_acc*100:.2f}%")

# ── Accuracy broken down per SNR ───────────────────────────────────────────
def eval_snr_curve(model, X, Y, snrs):
    out = {}
    for snr in sorted(np.unique(snrs)):
        idx = np.where(snrs == snr)[0]
        _, acc = model.evaluate(X[idx], Y[idx], verbose=0)
        out[snr] = acc
    return out

snr_curve = eval_snr_curve(model, Xte_n, Yte, SNRte)

plt.figure(figsize=(9, 5.5))
xs = sorted(snr_curve); ys = [snr_curve[x]*100 for x in xs]
plt.plot(xs, ys, marker='o', color='#DD8452', linewidth=2, markersize=6)
plt.axvline(0, color='gray', linestyle='--', alpha=0.6)
plt.title('Attention+CLDNN — Accuracy vs SNR (trained on all SNRs, AWGN)', fontweight='bold')
plt.xlabel('SNR (dB)'); plt.ylabel('Accuracy (%)')
plt.grid(alpha=0.35)
plt.tight_layout(); plt.show()

## 🏁 Step 9 — Accuracy at 0 dB (headline result)

In [ ]:
acc_0db = snr_curve.get(0, None)

print("=" * 52)
print("  ACCURACY AT 0 dB — Attention+CLDNN (all-SNR train)")
print("=" * 52)
if acc_0db is not None:
    print(f"  Accuracy @ 0 dB : {acc_0db*100:.2f}%")
else:
    print("  0 dB not found exactly in test SNRs — check SNR_raw values.")
print(f"  Overall accuracy (all SNRs) : {test_acc*100:.2f}%")
print("=" * 52)

## 🔎 Step 10 — Confusion Matrix at 0 dB

In [ ]:
idx_0 = np.where(SNRte == 0)[0]
y_pred = model.predict(Xte_n[idx_0], verbose=0)
y_true_lbl = np.argmax(Yte[idx_0], axis=1)
y_pred_lbl = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_true_lbl, y_pred_lbl, normalize='true')

plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=mods, yticklabels=mods, cbar=True)
plt.title('Attention+CLDNN — Confusion Matrix @ 0 dB', fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

print(f"\n✅ Done. Accuracy @ 0 dB = {acc_0db*100:.2f}%")